In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr, rand, col

# ダミーデータの生成
# 1,000行のベースデータを作成
raw_df = spark.range(1, 1001) \
    .withColumn("user_id", expr("int(rand() * 100) + 1000")) \
    .withColumn("amount", expr("int(rand() * 5000) + 500")) \
    .withColumn("category", expr("case when rand() > 0.5 then 'Electronics' else 'Books' end"))

# データの確認
display(raw_df.limit(5))

In [0]:
# カテゴリが 'Electronics' のデータのみに絞り込み、ユーザーごとに集計
aggregated_df = raw_df \
    .filter(col("category") == "Electronics") \
    .groupBy("user_id") \
    .sum("amount") \
    .withColumnRenamed("sum(amount)", "total_electronics_amount") \
    .orderBy("total_electronics_amount", ascending=False)

# 加工後データの確認
display(aggregated_df)

In [0]:
# ディスクではなく、メモリ上の一時ビュー（View）として登録
view_name = "hands_on_sales_summary"

aggregated_df.createOrReplaceTempView(view_name)

print(f"一時ビューの登録が完了しました。SQL等で '{view_name}' としてクエリ可能です。")

In [0]:
# 集計結果のTop 5を抽出して確認（可視化用の疑似処理）
top_users = aggregated_df.limit(5)
display(top_users)

# ※Databricksの画面上で、結果の下にある「＋」ボタンから「Visualization」を選び、
# 棒グラフ（Bar chart）などをサクッと作ってみると、より開発っぽくなります！